# Stage 0 - inspect (Kaggle, GPU T4 x2 or CPU, Internet ON)
Paste the full output of the last two cells back to the assistant. Expected time: 5-15 min (mostly the 2B SAE download, ~6.5 GB). **Untested.**

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

In [ ]:
!python scripts/01_check_paper_tables.py | tail -3
!python scripts/02_inspect_guise.py --guise-dir {GUISE} --authors-clone /tmp/REINS

In [ ]:
!python scripts/03_inspect_sae_ckpt.py --sae-root {SAE_ROOT} --layers 0,14,23 --d-model 2048 --width 16384

In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
print("transformers", transformers.__version__, "| gpus", torch.cuda.device_count())
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B-Base")
m = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-2B-Base", torch_dtype=torch.float16).cuda().eval()
import sys; sys.path.insert(0, "src")
from reinsmini import steer as S
print(S.find_layer_list(m, 24)[0], "| decoder blocks: 24")
ids = tok("Question: What is the capital of France?\n\nAnswer:", return_tensors="pt").to("cuda")
print(tok.decode(m.generate(**ids, max_new_tokens=30, do_sample=False)[0]))

In [ ]:
!rm -rf /tmp/REINS /tmp/sae